# Assignment 01 - Study Planner Agent

This notebook builds a simple **Study Planner Agent**. It can show the study topics it knows and return a short study plan for a selected topic.

The notebook follows the same basic structure as the lesson example:
1. create the model client;创建模型客户端
2. define tools;定义工具函数
3. create and invoke the agent;创建并调用智能体
4. stream a second response.流式输出第二次响应


## Step 1 - Set up the model client

This cell loads the model configuration from the repository's `.env` file. I do not put the API key, model name, or base URL directly in the notebook.


In [4]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")


Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## Step 2 - Define the tools

The first tool returns the list of study topics supported by this agent. 
返回该智能体支持的所有学习列表。
The second tool accepts a subject name and uses that argument to return a matching study plan.
接收一个科目名称参数，并用这个参数返回对应的学习计划。

In [6]:
#从langchain的工具模块导入tool装饰器，用于把普通函数包装成agent可用工具
from langchain.tools import tool

# tool装饰器：标记这个函数是智能体可以调用的工具
@tool
def get_study_subjects() -> list[str]:
    """Get the list of study subjects supported by the study planner."""
    return [
        "Python basics",     # Python基础
        "LangChain agents",  # LangChain智能体
        "Git and GitHub",    # Git与GitHub
        "Prompt engineering",# 提示词工程
    ]

# tool装饰器：将函数注册为Agent调用工具
@tool
def get_study_plan(subject: str) -> str:
    """Get a short study plan for a supported subject name."""
    # 字典：key为小写科目名，value是对应学习计划
    plans = {
        "python basics": (
            "Review variables and data types; practice if statements and loops; "
            "write two small functions; finish with a short debugging exercise."
        ),
        "langchain agents": (
            "Review the model-tool-agent relationship; study the @tool decorator; "
            "practice create_agent; inspect one tool call and its result."
        ),
        "git and github": (
            "Review status, add, commit, and push; practice creating a branch; "
            "make one small change and inspect the commit history."
        ),
        "prompt engineering": (
            "Review clear instructions and constraints; compare two prompt versions; "
            "test examples; record which wording gives the most reliable answer."
        ),
    }

    # 去除科目名称首尾空格，并全部转为小写，方便匹配字典key
    key = subject.strip().lower()

    # 判断科目不在支持列表内，返回提示信息
    if key not in plans:
        return (
            f"{subject} is not covered by the current study tools. "
            "Use get_study_subjects to see the supported topics."
        )

    # 匹配成功，返回对应的学习计划文本
    return plans[key]


In [7]:
print("llm exists:", "llm" in globals())

llm exists: True


In [8]:
print(llm)
print(get_study_subjects)
print(get_study_plan)

metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-openai': '1.6.2'}} client=<openai.resources.chat.completions.completions.Completions object at 0x72c8b9f1f140> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x72c8b9b09ca0> root_client=<openai.OpenAI object at 0x72c8bbf52ea0> root_async_client=<openai.AsyncOpenAI object at 0x72c8ba6e1970> model_name='deepseek-v4-pro' model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='https://api.deepseek.com/v1' stream_chunk_timeout=120.0 extra_body={}
name='get_study_subjects' description='Get the list of study subjects supported by the study planner.' args_schema=<class 'langchain_core.utils.pydantic.get_study_subjects'> func=<function get_study_subjects at 0x72c8c0102520>
name='get_study_plan' description='Get a short study plan for a supported subject name.' args_schema=<class 'langchain_core.utils.pydantic.get_study_plan'> func=<function get_study_plan

## Step 3 - Create the agent

I connect the model and both tools with `create_agent`. The system prompt tells the agent what it does and tells it to use the tools for supported study information instead of inventing plans.


In [9]:
# 从langchain智能体模块导入create_agent函数，用来组装agent
from langchain.agents import create_agent

# 创建智能体实例
agent = create_agent(
    llm, # 传入之前初始化好的大语言模型客户端
    # 传入我们刚刚定义好的两个工具函数
    tools=[get_study_subjects, get_study_plan],
    # 系统提示词：给智能体设定角色与行为规则
    system_prompt=(
        "You are a study planning assistant. Help users choose a supported study topic "
        "and make a short study plan. Use get_study_subjects when the user asks what "
        "topics are available. Use get_study_plan when the user asks for a plan for a "
        "specific subject. Use the tools instead of guessing supported topics or plans. "
        "If a requested subject is not covered by the tools, say that clearly."
    ),
)


## Step 4 - Invoke the agent and inspect the whole message history

This question should make the agent use the tools. I print every message so the user message, model tool call, tool result, and final model answer can be inspected.


In [ ]:
# 调用智能体，传入用户提问，得到执行结果
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user", # 用户角色消息
                "content": (
                    # 用户提问：有哪些可用学习主题？今晚我想复习LangChain智能体，请给我这个主题的学习计划。
                    "What study topics are available? I want to review LangChain agents "
                    "this evening, so please give me a plan for that topic."
                ),
            }
        ]
    }
)

# 遍历结果里所有消息，逐条打印查看完整对话链路
for message in result["messages"]:
    print(f"{message.type}:")
    # 判断当前消息是否包含工具调用信息
    if getattr(message, "tool_calls", None):
        print("tool_calls:", message.tool_calls)
    # 打印消息正文内容
    print(message.content)
    # 打印分割线，方便区分不同消息
    print("-" * 60)


human:
What study topics are available? I want to review LangChain agents this evening, so please give me a plan for that topic.
------------------------------------------------------------
ai:
tool_calls: [{'name': 'get_study_subjects', 'args': {}, 'id': 'call_00_G8sVbhBwSTOrfHJq6qdM0671', 'type': 'tool_call'}]

------------------------------------------------------------
tool:
["Python basics", "LangChain agents", "Git and GitHub", "Prompt engineering"]
------------------------------------------------------------
ai:
tool_calls: [{'name': 'get_study_plan', 'args': {'subject': 'LangChain agents'}, 'id': 'call_00_I9r0A8LlmJmG9rdG2V015206', 'type': 'tool_call'}]

------------------------------------------------------------
tool:
Review the model-tool-agent relationship; study the @tool decorator; practice create_agent; inspect one tool call and its result.
------------------------------------------------------------
ai:
Here are the study topics currently supported:

- Python basics
- L

## Step 5 - Stream a second question

For the second question, I use `astream(..., stream_mode="messages")`. Only chunks produced by the model node are printed, so the answer appears progressively instead of as one final block.


In [ ]:
# 异步遍历 agent.astream 产生的流式输出，每次拿到token片段和元数据
async for token, metadata in agent.astream(
    {
        "messages": [
            {
                "role": "user",
                # 用户提问：请生成Git和GitHub的学习计划
                "content": "Please make a study plan for Git and GitHub.",
            }
        ]
    },
    stream_mode="messages", # 流式模式：按消息粒度返回片段
):
    # 判断：当前节点是model模型节点，并且token存在content文本内容
    if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
        # 打印流式文本片段，end=""不换行，flush=True强制立刻输出，实现打字机效果
        print(token.content, end="", flush=True)
print()


Here's a short study plan for **Git and GitHub**:

1. **Review status, add, commit, and push** — get comfortable with the core workflow: `git status`, `git add`, `git commit`, and `git push`.
2. **Practice creating a branch** — learn how to create and switch branches with `git branch` and `git checkout` (or `git switch`).
3. **Make one small change and inspect the commit history** — modify a file, commit it, and review the log with `git log` to understand what was recorded.

This plan is designed to build your confidence with the everyday Git/GitHub workflow. Good luck!


## Extra behavior checks

I also try one general study question that does not need the course tools and one unsupported subject. These checks help me see when the model answers directly and how it behaves when the tools do not contain the requested information.


In [ ]:
# ========== 额外行为测试 ==========
# 测试1：通用问题，无需调用自定义工具，模型直接作答
direct_result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is one good habit for staying focused while studying?"}]}
)
# [-1]取消息数组最后一条，也就是AI最终回复
print("General question:")
print(direct_result["messages"][-1].content)

# 测试2：请求不支持的科目（微积分），验证agent的错误提示逻辑
unsupported_result = agent.invoke(
    {"messages": [{"role": "user", "content": "Make me a study plan for calculus."}]}
)
print("\nUnsupported subject:")
print(unsupported_result["messages"][-1].content)


General question:
One good habit is **using the Pomodoro Technique**: study in focused blocks of about 25 minutes, then take a 5-minute break.

This works well because:

- A short, timed session feels more manageable than "studying all afternoon."
- Knowing a break is coming makes it easier to resist distractions.
- Regular breaks help keep your mind fresh and improve long-term focus.

If 25 minutes feels too long or too short, you can adjust the intervals—for example, 45 minutes of study with a 10-minute break.

Unsupported subject:
I can't create a study plan for **calculus**, because it isn't one of the topics supported by the study planner right now.

Here are the topics I *can* make a plan for:

- Python basics
- LangChain agents
- Git and GitHub
- Prompt engineering

Would you like me to make a study plan for any of these instead?


## Reflection

**1. When did the model call a tool?**

The model called tools when I asked what study topics were available and when I asked for a study plan for LangChain agents. It first used `get_study_subjects()` to get the list of supported topics, and then used `get_study_plan()` to get the study plan for LangChain agents.

**2. When did the model answer without using a tool?**

The model answered without using a tool when I asked a general question such as “What is one good habit for staying focused while studying?” This question did not depend on the information stored in the tools, so the model could answer it directly.

**3. What happened when I asked about something the tools do not cover?**

When I asked for a study plan for calculus, the tools did not contain that subject. The agent explained that calculus was not supported by the current study tools instead of making up a study plan.


**1. 模型什么时候调用了工具？**

当我询问有哪些学习主题，以及要求制定 LangChain agents 的学习计划时，模型调用了工具。它先使用 get_study_subjects() 获取可学习主题列表，再使用 get_study_plan() 获取 LangChain agents 的具体学习计划。

**2. 模型什么时候没有调用工具，而是直接回答？**

当我询问类似“学习时保持专注的一个好习惯是什么？”这样的通用问题时，模型没有调用工具，因为这个问题不依赖工具中保存的信息，可以直接回答。

**3. 当我询问工具没有覆盖的内容时发生了什么？**

当我要求制定 calculus（微积分）的学习计划时，工具中没有这个主题，因此智能体说明当前工具不支持微积分，而没有自己编造一个学习计划。
